In [ ]:
import folium, datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq
from glob import glob
from os.path import join
from multiprocessing import Pool
from skmob.core.trajectorydataframe import TrajDataFrame
from skmob.measures import individual
from skmob.preprocessing import filtering, compression, detection, clustering
import branca.colormap as cm
import matplotlib.pyplot as plt

In [ ]:
data_path = ""
data = pd.read_parquet(data_path, engine='pyarrow')

In [ ]:
path_zips = "/glade/work/cbecker/california_zip.geojson"
gdf_zips = gpd.read_file(path_zips)

In [ ]:
gdf_zips_counts = pd.merge(gdf_zips, data.groupby("zipcode").size().rename("count"), left_on="ZIP_CODE", right_on="zipcode")
gdf_zips_counts.head()

In [ ]:
hist, bins, _ = plt.hist(np.sort(gdf_zips_counts["count"]), bins='fd')
plt.show()

In [ ]:
vmax = gdf_zips_counts["count"].max()
linear = cm.linear.GnBu_03.to_step(n=8, data=gdf_zips_counts["count"], method="log")
tooltips = ["ZIP_CODE", "PO_NAME", "POPULATION", "POP_SQMI", "count"]

m = folium.Map(location=[34.0549, -118.2426],
               zoom_start=8)

folium.GeoJson(
    gdf_zips_counts,
    style_function=lambda feature: {
        "fillColor": linear(feature["properties"]["count"]),
        "fillOpacity": 0.9,
        "color": "black",
        "weight": 0.5,
    },
    tooltip=folium.features.GeoJsonTooltip(tooltips),
).add_to(m)

m.save("home_locations.html")
m

![title](img/hls.png)

In [ ]:
gdf_zips_counts["hldivpop"] = gdf_zips_counts["count"] / np.maximum(gdf_zips_counts["POPULATION"], 1)

In [ ]:
plt.hist(gdf_zips_counts["hldivpop"], bins="auto")
plt.show()

In [ ]:
vmax = gdf_zips_counts["count"].max()
linear = cm.LinearColormap(colors=["blue", "green", "yellow", "orange", "red"], index=[0, 1, 2, 100, gdf_zips_counts["hldivpop"].max()])
tooltips = ["ZIP_CODE", "PO_NAME", "POPULATION", "POP_SQMI", "count", "hldivpop"]

m = folium.Map(location=[34.0549, -118.2426],
               zoom_start=8)

folium.GeoJson(
    gdf_zips_counts,
    style_function=lambda feature: {
        "fillColor": linear(feature["properties"]["hldivpop"]),
        "fillOpacity": 0.6,
        "color": "black",
        "weight": 0.4,
    },
    tooltip=folium.features.GeoJsonTooltip(tooltips),
).add_to(m)

m.save("home_locations_div_pop.html")
m

![title](img/hlsvspop.png)